# Gen3 Embeddings Demo

> This will demonstrate how to create and retrieve embeddings in bulk from Gen3

First, let's install the Gen3 Python Software Development Kit (SDK), which includes a command line interface (CLI).

In [ ]:
%pip install --upgrade pip
# %pip install gen3 --upgrade

In [ ]:
# we also need pandas for some nice visualizations of output files
import pandas as pd

If you are running Gen3 locally, you can set some variables to point to the right credentials file.

If you are trying to interact with a production instance, just leave the default `ai`. e.g. don't uncomment - the default credentials setup should point you to the right place if you have your API key in the default location `~/.gen3/credentials.json`.

In [ ]:
url_prefix = "https://foobar.dev.planx-pla.net/ai"
auth = "--auth ~/.gen3/local_helm_test_user.json"

# url_prefix = "https://foobar-local.dev.planx-pla.net/ai"
# auth = ""

## Gen3 AI Embeddings CLI

In [ ]:
!gen3 ai embeddings --help

In [ ]:
!gen3 ai embeddings collections --help

Get a response from the service by reading collections. 

> IMPORTANT: You need appropriate permissions to read (and for future sections: write).
> So these commands may be empty or fail unless you have those authorizations in the environment your credentials are for.

In [ ]:
!gen3 $auth ai embeddings collections read

## Create Embeddings Collections

This will show creation of collections and embeddings. So you need write permission or this will fail. You can run the Gen3 Embeddings API locally to test this out. See the [Gen3 AI repo README](https://github.com/uc-cdis/gen3-ai) for more information.

In [ ]:
!gen3 $auth ai embeddings collections delete "ctds-github-md"
!gen3 $auth ai embeddings collections create "ctds-github-md" --dimensions 384 --description "All markdown from CTDS Github"

In [ ]:
!gen3 $auth ai embeddings collections read "ctds-github-md" 

In [ ]:
# try to delete collections that might already exist
!gen3 $auth ai embeddings collections delete "test_expr"
!gen3 $auth ai embeddings collections delete "test_hist"
!gen3 $auth ai embeddings collections delete "test_summ"

Let's create some more example collections.

> IMPORTANT: You need permission to create and manage these collections *before* running the commands. So ensure the Gen3 operator adds these resources to the `user.yaml` and provides your user permission to them. If you are running Gen3 yourself, you can see the [Gen3 AI repo README](https://github.com/uc-cdis/gen3-ai) for more information on how to set up appropriate auth.

In [ ]:
!gen3 $auth ai embeddings collections create "test_expr" --dimensions 256 --description "test expr data"
!gen3 $auth ai embeddings collections create "test_hist" --dimensions 1536 --description "test hist data"

You can also create collections of larger dimensional size. This will use a `vector_type` of `halfvec` to fit it into the underlying database.

In [ ]:
!gen3 $auth ai embeddings collections create "test_summ" --dimensions 4096 --description "test summ data"

## Publish Data into Embeddings Collections

Now that we have created collections for embeddings, we can publish the actual embeddings into those collections.

To do this, you need a Gen3 Embeddings Manifests. Conveniently, there are examples in the Gen3 Python SDK/CLI repo in the tests folder you can use.

In [ ]:
!gen3 ai embeddings publish --help

See the above help message to understand what we need to publish embeddings. 

**tl;dr** we need a manifest file with a row per embedding. That row needs to contain the vector (embedding), along with other metadata.

In [ ]:
# here's a quick visualization of the columns/data of this input manifest
df = pd.read_csv('../../tests/embeddings_tests/test_expr.tsv', sep='\t', nrows=10)
df

In [ ]:
!gen3 $auth ai embeddings publish ../../tests/embeddings_tests/test_expr.tsv --default-collection test_expr --batch-size 50

In [ ]:
# here's a quick visualization of the columns/data of this input manifest
df = pd.read_csv('../../tests/embeddings_tests/test_hist.tsv', sep='\t', nrows=10)
df

In [ ]:
!gen3 $auth ai embeddings publish ../../tests/embeddings_tests/test_hist.tsv --default-collection test_hist --batch-size 20

In [ ]:
# here's a quick visualization of the columns/data of this input manifest
df = pd.read_csv('../../tests/embeddings_tests/test_summ.tsv', sep='\t', nrows=10)
df

In [ ]:
!gen3 $auth ai embeddings publish ../../tests/embeddings_tests/test_summ.tsv --default-collection test_summ --batch-size 10 --overwrite

## Convert Published Embeddings Manifests into Indexing Manifests

Each of the `publish` commands above generated an output file `{input_filename}_output.tsv` (unless you overrode the output filename). 

Those outputs are manifests that now contain the final `embedding_id` and other Gen3 Embedding information that came back from creating embeddings through the Gen3 Embeddings API.

We can **convert** those _outputted_ Published Gen3 Embeddings Manifests into Gen3 Indexing Manifests to _input_ into the Gen3 Indexing process. This will allow us to create persistent, indexed records with globally unique identifiers (GUIDs) in Gen3 through the Gen3 Indexing API.

So first, let's convert to the expected format for indexing.

In [ ]:
# here's a quick visualization of the columns/data of the output manifest from previous `publish` commands
# this is what we'll convert
df = pd.read_csv('../../tests/embeddings_tests/test_expr_output.tsv', sep='\t', nrows=3)
df

In [ ]:
!gen3 ai embeddings convert --help

In [ ]:
!gen3 $auth ai embeddings convert ../../tests/embeddings_tests/test_expr_output.tsv --url-prefix $url_prefix
!gen3 $auth ai embeddings convert ../../tests/embeddings_tests/test_hist_output.tsv --url-prefix $url_prefix
!gen3 $auth ai embeddings convert ../../tests/embeddings_tests/test_summ_output.tsv --url-prefix $url_prefix

That `convert` command created new `{original_filename}_converted.tsv` files. Let's take a look at those:

In [ ]:
df = pd.read_csv('../../tests/embeddings_tests/test_expr_output_converted.tsv', sep='\t', nrows=3)
df

## Create Gen3 Indexed Records with the Indexing Manifest

The manifest we converted to above is a Gen3 Indexing Manifest with no GUIDs - we'll let the Gen3 Indexing command and backend generate those for us.

The `md5` checksum and `size` in bytes columns above are of the JSON-stringified version of the vector (in other words, the "data" is the vector itself).

`url` is a direct link to the Gen3 Embeddings API for that particular embedding.

Now, before we index, let's validate our Gen3 Indexing manifest is correctly formatted:

In [ ]:
!gen3 objects manifest validate-manifest-format --help

In [ ]:
!gen3 objects manifest validate-manifest-format ../../tests/embeddings_tests/test_expr_output_converted.tsv --allowed-protocols "https http"
!gen3 objects manifest validate-manifest-format ../../tests/embeddings_tests/test_hist_output_converted.tsv --allowed-protocols "https http"
!gen3 objects manifest validate-manifest-format ../../tests/embeddings_tests/test_summ_output_converted.tsv --allowed-protocols "https http"

In [ ]:
!gen3 objects manifest publish --help

> IMPORTANT NOTE: Publishing the manifest through the Gen3 Indexing API requires permissions to write to that API. 

**For Gen3 Operators**: typically there is an `indexd_admin` policy. One option to provide the necessary permissions is to add the `/vectorstore/collection/` resource path to that. This will allow full administrative management of indexing records with `authz` beginning with that path.

Example `user.yaml` snippet:

```yaml
...
      - id: indexd_admin
        description: full access to indexd API
        role_ids:
          - indexd_admin
        resource_paths:
          - /programs
          - /vectorstore/collections       <----- THIS IS NEW
...
```

In [ ]:
!gen3 $auth -v objects manifest publish ../../tests/embeddings_tests/test_expr_output_converted.tsv --out-manifest-file ../../tests/embeddings_tests/test_expr_output_converted_indexed.tsv --thread-num 1

In [ ]:
!gen3 $auth -v objects manifest verify ../../tests/embeddings_tests/test_expr_output_converted_indexed.tsv --max-concurrent-requests 1

> Note: if the above command fails, you can try to move on anyway. It is just to double-check that the manifest entries got created - which may be true even if the verify command fails. We plan on addressing this flakiness in a future update.

In [ ]:
!gen3 $auth -v objects manifest publish ../../tests/embeddings_tests/test_hist_output_converted.tsv --out-manifest-file ../../tests/embeddings_tests/test_hist_output_converted_indexed.tsv --thread-num 1

In [ ]:
!gen3 $auth -v objects manifest verify ../../tests/embeddings_tests/test_hist_output_converted_indexed.tsv --max-concurrent-requests 1

In [ ]:
!gen3 $auth -v objects manifest publish ../../tests/embeddings_tests/test_summ_output_converted.tsv --out-manifest-file ../../tests/embeddings_tests/test_summ_output_converted_indexed.tsv --thread-num 1

In [ ]:
!gen3 $auth -v objects manifest verify ../../tests/embeddings_tests/test_summ_output_converted_indexed.tsv --max-concurrent-requests 1

Now there are indexed records with GUIDs. The above tool output an `*_indexed.tsv` manifest. We can take a look and see that now the `guid` column is filled out! 

In [ ]:
df = pd.read_csv('../../tests/embeddings_tests/test_expr_output_converted_indexed.tsv', sep='\t', nrows=3)
df

## Everything is in Gen3 now!

Let's recap. We have:

- Created Gen3 Embeddings Collections to store embeddings of different dimensionality
- Ingested Embeddings Manifests of embeddings through the Gen3 Embeddings API into the collections we created
- Converted output manifest from embedding creation into indexing manifests
- Ingested Indexing Manifests to create Gen3 Indexed Records with assigned GUIDs

At this point, we have the embeddings themselves stored and we've created persistent identifiers (GUIDs). 

Now you can use those GUIDs the same way you use other Gen3 GUIDs for files. So if you have a Gen3 Data Model with nodes that point to sample files identified by GUIDs, you can now have a new "embedding" node with a GUID pointing to the embedding!

This will allow traditional Gen3 search for data with embeddings.

If you want to search the embeddings collections themselves, the Gen3 Embedding API exposes search functionality as well. This supports similarity search.

## Bulk Retrieval of Gen3 Embeddings via their Indexed GUIDs

Let's assume we've found GUIDs of interest via a search in Gen3 (and for simplicity, let's assume those GUIDs are ALL the GUIDs we have indexed previously in this notebook). Now, given **only** those GUIDs, we want to bulk retrieve the actual embeddings from Gen3 to do some AI/ML analysis.

In [ ]:
"""
This just aggregates all the GUIDs from all our indexed files into a single file.

This output file simulates (or rather, skips) the process of finding data of interest and getting the GUIDs.
"""
import glob
import pandas as pd

file_pattern = "../../tests/embeddings_tests/test_*_output_converted_indexed.tsv"
output_file = "../../tests/embeddings_tests/test_aggregated_guids.tsv"

all_files = glob.glob(file_pattern)

df_list = []
for file in all_files:
    # usecols ensures we don't waste memory. only load guids
    df = pd.read_csv(file, sep="\t", usecols=["guid"])
    df_list.append(df)

combined_df = pd.concat(df_list, ignore_index=True)
combined_df.to_csv(output_file, sep="\t", index=False)

print(f"Done! Aggregated {len(all_files)} files into: {output_file}")

In [ ]:
df = pd.read_csv('../../tests/embeddings_tests/test_aggregated_guids.tsv', sep='\t')
df

## Bulk retrieve embeddings from Fence

Now we can pretend we found a bunch of GUIDs of interest in Gen3.

For this example, we'll use all the GUIDs we previously created.

By providing a manifest of only GUIDs (or passing on the CLI), we can get the _content_ of the referenced data from the indexed record, e.g. the embedding itself.


In [ ]:
from gen3.file import Gen3File
from gen3.auth import Gen3Auth
import pandas as pd

import time

start = time.perf_counter()

#auth = Gen3Auth(refresh_file="FULL_PATH_TO/creds.json")
auth = Gen3Auth(refresh_file="FULL_PATH_TO/local_helm_test_user.json")
gen3_file = Gen3File(auth.endpoint, auth_provider=auth)

embeddings_contents = gen3_file.get_bulk_content(input_file="../../tests/embeddings_tests/test_aggregated_guids.tsv")
 
end = time.perf_counter()
print(f"Total runtime: {end - start:.3f} seconds")

print(f"Got all {len(embeddings_contents)} GUIDs")

rows = []
for i, (guid, emb) in enumerate(embeddings_contents.items()):
    if i >= 100:  # sample first 100 to keep it light
        break
    rows.append({
        "guid": emb.guid,
        "embedding_id": emb.embedding_id,
        "embedding_len": emb.embedding.shape[0],
        "dtype": str(emb.embedding.dtype),
        "authz": emb.authz,
        "collection_id": emb.collection_id,
        **emb.metadata
    })

df_sample = pd.DataFrame(rows)
df_sample.head()

# for guid, embedding_content in embeddings_contents.items():
#     # the GUIDs are already an efficient numpy array!
#     print(type(embedding_content.embedding))
#     print(embedding_content)
#     break


# do other AI/ML work with the embeddings!

We have shown the full flow from original vectors -> storage in Gen3 -> retrieval from Gen3 in an efficient bulk pipeline. 

We've also shown that Gen3 Embeddings can be indexed like files to provide GUIDs which can be referenced and searched the same way file-based GUIDs are today.

## Search

In [ ]:
from gen3.ai import EmbeddingsClient
from gen3.auth import Gen3Auth
import pandas as pd

auth = Gen3Auth(refresh_file="FULL_PATH_TO/local_helm_test_user.json")
endpoint = "https://foobar.dev.planx-pla.net/ai"
embeddings_client = EmbeddingsClient(auth=auth, endpoint=endpoint)

input_embedding = [0.8378045397688774, 0.06987507157848805, 0.703522778584545, 0.667387500123993, 0.8995997785594614, 0.5564736769747567, 0.8784417090078025, 0.832738087896886, 0.3851110139704601, 0.45045646017351504, 0.4496777540415481, 0.8504620149597644, 0.7678266900757573, 0.9833677581198345, 0.7879340052581356, 0.542592271356153, 0.9119705880016692, 0.34718618661835143, 0.051991856134469105, 0.34044360659835393, 0.2746126996173005, 0.9172220131856329, 0.7567776544694546, 0.23217972293290323, 0.741919237711306, 0.4627282417177442, 0.16655125567288642, 0.335407180970943, 0.9776187998884746, 0.3777673681795587, 0.8333444953843695, 0.05506358533155831, 0.6199937913424716, 0.9039776474198628, 0.8831114636150207, 0.8048522109493741, 0.19282102990851324, 0.36923992967465047, 0.2891814328682707, 0.6411141144934405, 0.8361002844929453, 0.9388482509511638, 0.09988468026681274, 0.2394332320184026, 0.9385146659914949, 0.07440330421193975, 0.310006173070003, 0.27854960738917534, 0.4954076988367907, 0.06967598217499393, 0.7317615305166378, 0.26023676239674287, 0.31576011695121053, 0.9578348331545626, 0.7790263801020981, 0.22125634269993544, 0.875221261645208, 0.9381462204346596, 0.4213965486294383, 0.251856219485592, 0.27155047733792903, 0.7218309068180636, 0.7932211899173107, 0.801768075485567, 0.508021166003396, 0.998819412422052, 0.2639459034535674, 0.5924193082456456, 0.6868229635164657, 0.7528612757457608, 0.4693696318959907, 0.514584020964411, 0.6989967722449594, 0.5354007406171354, 0.8152510760523148, 0.9122312786637787, 0.8417846264993416, 0.8986522248482539, 0.0506625194624345, 0.7802180848973225, 0.41584566472494533, 0.22431443874855406, 0.8524764436476022, 0.9111515895071044, 0.8840088870839792, 0.844869682703705, 0.10274678629246592, 0.2438403817088216, 0.18375647029357245, 0.8539170772392283, 0.6863873549659344, 0.0709082747387556, 0.8157442868677136, 0.9564354084846073, 0.6867398922623854, 0.43729611930382306, 0.7590570446650251, 0.2553353884720715, 0.4287544531729671, 0.5380069740866767, 0.37715513932002875, 0.14866708893397262, 0.5645599040262639, 0.019809110341296732, 0.17615771849018846, 0.7247913736568556, 0.37388420979116543, 0.7205947256914034, 0.022141145323811395, 0.7738395863044804, 0.8760092949258261, 0.02128945497318724, 0.6689529633743628, 0.1481141832697791, 0.7476803487081487,0.7090524037654332, 0.028567409758832096, 0.679164663354716, 0.8426618670714402, 0.0960149899875472, 0.9706166057220322, 0.7686330470907047, 0.9518538977718236, 0.911855697356858, 0.05062035472148596, 0.00010475919835661873, 0.29326499739012857, 0.3829089367528682, 0.3771424095921394, 0.502827068095375, 0.17858202827131386, 0.9210024303680351, 0.11207586380846768, 0.3189570815063544, 0.4500324566214703, 0.07367341171001895, 0.6293656959734538, 0.9667704317797506, 0.926761653338055, 0.36855930150517113, 0.8158750241441288, 0.36347802653130923, 0.3569432230726557, 0.6263266898688996, 0.5214323849921562, 0.4775073787990939, 0.932378181970079, 0.34322882720915016, 0.33535003234049476, 0.4731185002384162, 0.49825844033650624, 0.6499517106486734, 0.873666890604586, 0.3466301029006761, 0.466036939514066, 0.5479434144219715, 0.44909881780476346, 0.9290089880397554,0.4069468586799512, 0.017353726995371965, 0.1111451026516237, 0.7264621883955908, 0.7776773227234914, 0.29197619717275725, 0.22450575419519236, 0.5881403827540048, 0.08014840501472908, 0.9324387725398354, 0.5631457489261892, 0.2119564761846523, 0.966362722363462, 0.7261044244428156, 0.28230456722931396, 0.7984087985313054, 0.2866809471070837, 0.13659456035840112, 0.53264126702573, 0.26854791784350296, 0.11634852210364321, 0.511572087239187, 0.16327026853829085, 0.898269826758326, 0.6622414695489429, 0.15637221637889864, 0.3384625506228831, 0.8885793515237851, 0.004798430503774265, 0.5290776207136603, 0.9241267599320402, 0.4007711456186991, 0.48730887057792904, 0.443326558604922, 0.8579277970421718, 0.3778492020307801, 0.15272351841043086, 0.6308127529993167, 0.370055711079719, 0.5972220229873405, 0.30052353794034425, 0.013455942646796615, 0.3288786428449, 0.3084299519835866, 0.3199943844684049, 0.23739684933669103, 0.8759815951388583, 0.28599693922583025, 0.23131657812708784, 0.9590830581617201, 0.46327752348781714, 0.8459706238500595, 0.8851771425152873, 0.18882893635056885, 0.5711186870704813, 0.31851337919540446, 0.02472382924836003, 0.6360477991664595, 0.777535931743049, 0.6781238016523574, 0.8554177698207145, 0.05216155507554776, 0.9252728767121488, 0.9530105226722775, 0.7449666733430375, 0.405407265603436, 0.5527472109354155, 0.7987933466626148, 0.191621142407421, 0.9913248984982096, 0.6490971846338945, 0.44440986236242586, 0.9207751813252364, 0.6107055581120949, 0.6032978356088382, 0.17557129057736076, 0.8778039597609985, 0.5493867256972993, 0.4478342209806768,0.7652201761693814, 0.1141097703827586, 0.8008851447280777, 0.4212062531964599, 0.3011764946139973, 0.00525131897721165, 0.3092299740156015, 0.6386455179979693, 0.6509017399267285, 0.5077207810756812, 0.3496615815822962, 0.7638247226671437, 0.08157520779932947, 0.37631778275011774, 0.6375712438760012, 0.4958684666583415, 0.8998096523699194, 0.4101707786520489, 0.6466832783045998, 0.6427415257952719, 0.7074210467560557, 0.8193887231494071, 0.23272225038537164, 0.2029086023486938, 0.7064462082044116, 0.4522163618555024, 0.6716176499599911, 0.6709806464938112, 0.6392574088730195, 0.7452298101433046, 0.6073600290618998, 0.6866319678257164, 0.34462986796640527, 0.8538116627186976, 0.6596473846154516, 0.08931008905714954, 0.9540484846026512, 0.6621547516520311, 0.8964324733218515, 0.16506672658730948, 0.5298871877486058, 0.1984806206426043, 0.9055860766564297, 0.12169641540181064, 0.4877457206425898, 0.619732533270732, 0.3033619835113449, 0.24449624791606372, 0.5628178067489297, 0.7669361047437091, 0.6265699488165802, 0.09179442265860982, 0.41783490744259444, 0.20167441767328387, 0.19405418565612043, 0.625610997296658, 0.7931762243884389, 0.04334976901666132, 0.8657173943621909, 0.6863004213460977, 0.4836224633715269, 0.6075892846772705, 0.07747766053949157, 0.02697532812829251, 0.45794523527836595, 0.9775823398184841, 0.9493665483330302, 0.7165602534240145, 0.30167789510542964, 0.6005196956815638, 0.601821419366173, 0.8885471847260122, 0.24171099953669484, 0.7957186826070273, 0.4812184379562915, 0.9439878491621001, 0.29555693757738066, 0.26259765550904446, 0.30822612896930934, 0.8384309145861315, 0.9270156661238697, 0.9248953819352773, 0.3695631602978686, 0.20055674893576814, 0.0742715370524456, 0.7702860922976624,0.38187653745553374, 0.04075093996191104, 0.9193807350913884, 0.23257433072140332, 0.8144094699158204, 0.22038551721525124, 0.6375710052900215, 0.4335142304349989, 0.6023936584908295, 0.19026700439060162, 0.08349851684219267, 0.8402568625005032, 0.6581104762373391, 0.6465319983892213, 0.7971833050164193, 0.8300383054212751, 0.025861130721398506, 0.03331637155977196, 0.6299393692816347, 0.28322636526519107, 0.3075885306542605, 0.5509413800328711, 0.4438074469299841, 0.9827065968041397, 0.18564719386384576, 0.739662210199604, 0.6293543243281277, 0.45266265444993925, 0.6984750468513862, 0.28005131907542935, 0.9077086824220183, 0.9613851377710829, 0.6082241430328096, 0.24375121613631745, 0.5141768231589351, 0.15379790932224213, 0.703889903672154, 0.8062655998714074, 0.9056278525020905, 0.6700251924843952, 0.4465047690730237, 0.4812547578530124, 0.10571739126399238, 0.9934715012353039, 0.20692193190072017, 0.7231983952704681, 0.8342004918906831, 0.22798834522411093, 0.7605700283044659, 0.8225859888356191, 0.08654716154374853, 0.9983891206242714, 0.3791722785092265, 0.06745267814841416, 0.11551188593395678, 0.09703454106996301, 0.9086484358756819,0.5621221683770621, 0.7937049035482434, 0.8275008334529474, 0.5570266475331592, 0.9968501453037126, 0.20926634814548795, 0.6568065674889586, 0.01280892260970723, 0.7740435012103978, 0.20742562349099136, 0.712245363258212, 0.05152022085828167, 0.4958710850901976, 0.33794713300177703, 0.4793921258866708, 0.3686743117110607, 0.09227238959307515, 0.5982628732085588, 0.5031320333134297, 0.9862985525743054, 0.13074812423434468, 0.38206784178305797, 0.12392015452848848, 0.33873526026272116, 0.20704056128137982, 0.3555109978509591, 0.8015892266039588, 0.9705831640565629, 0.12062534105487899, 0.7207377139339203, 0.2882056294738141, 0.5558250100439219, 0.11555731523845725, 0.5656537429317501, 0.4302836982487994, 0.688215471554617, 0.05496872326952995, 0.8162406701371159, 0.581926237623177, 0.1360506477748975, 0.33763018673437706, 0.7780009257799645, 0.3228050342937302, 0.1759697801424357, 0.7403496482599224, 0.9653197806910235, 0.12246269733822501, 0.9882399898786353, 0.48481904480211735, 0.14194406193435372,0.7168589828079194, 0.32950126311711436, 0.24953294433413353, 0.4559272725332557, 0.525991434939979, 0.7355113661737451, 0.6313857157965304, 0.47790202819365124, 0.1729696774080024, 0.810506607570538, 0.2821418809675451, 0.710884741888755, 0.12232324531619143, 0.7190661456119288, 0.8833083972236101, 0.8733586031928098, 0.1068936668164241, 0.25816478753630123, 0.5686187413796708, 0.06362851909837564, 0.2846546751383263, 0.9147808300548729, 0.33999076079370416, 0.8488794031452997, 0.25401915399923125, 0.07951034226789766, 0.11946463059564871, 0.578925127838502, 0.9911192050240976, 0.8667883125243556, 0.07260925023253195, 0.5167397448246257, 0.8163296108854511, 0.689757491344078, 0.8615016505191259, 0.9620008444324972, 0.34902720555399636, 0.27868740250085855, 0.282244580417352, 0.6163164091805089, 0.6989594457426942, 0.5203977353295771, 0.46568905199853117, 0.08195937970380363, 0.753255676673564, 0.8950332985364267, 0.33785513182187954, 0.35111222456271596, 0.9856887826642803, 0.8019312513844546, 0.0379034324906129, 0.41253806379997193, 0.10816282319094772, 0.49603076752318565, 0.808043701627366, 0.9715803902756737, 0.7178027380454027, 0.8939724463811328, 0.7967450810969121, 0.8941297723062774, 0.9347654498201641, 0.6259208806827061, 0.12754351806515285, 0.6255605245840311, 0.3285939203321796, 0.7311147190750336, 0.3401502048303807, 0.2959508811083482, 0.3173037984041369, 0.17972512166623933, 0.2240726455282257, 0.3220554935577633, 0.6633593129810854, 0.9098904690924083, 0.201232678988584, 0.15277942531938649, 0.2965086062080876, 0.5289283159591143, 0.32915262752335006,0.19704027325994444, 0.26231495731356336, 0.8231978887284366, 0.05644854039674063, 0.6991018543041185, 0.6843992748753839, 0.5545796764643129, 0.6807793487964747, 0.21954357471436514, 0.7341285503830721, 0.9431793319753993, 0.44416818668723135, 0.3531397921310543, 0.4294127825209795, 0.4283378663723134, 0.3525656306487893, 0.9377747256231769, 0.6957782011610489, 0.5612616690483253, 0.29847921490213636, 0.942013909757299, 0.06390837383364734, 0.0034352079690274095, 0.1414501226331073, 0.11904449687878438, 0.3763380022173314, 0.04810595739651957, 0.9591104605117302, 0.9365801958211937, 0.2738482270810433, 0.7567779929237319, 0.12042630073686145, 0.8719139917175479, 0.705933878021582, 0.3772795406762498, 0.9179719698793112, 0.2475778253778217, 0.6113865665969215, 0.014787791961175856, 0.26311502243422413, 0.8229624490547769, 0.22984429665559059, 0.7503307885011007, 0.1370311508799964, 0.28790449195894774, 0.06629047273293531, 0.5400470168387085, 0.5138086205829631, 0.10593122845327752, 0.3651395972552207, 0.8637298225218525, 0.5561135141985306, 0.649147846091654, 0.06510393732720887, 0.6143323021415119, 0.7401376997407458, 0.3206644874667768, 0.30935712304533547, 0.8319823061748644, 0.24726898697791289, 0.10193564771428842, 0.6034160551059703, 0.510638835550989, 0.4438548155554255, 0.457722410171431, 0.24283247187332335, 0.5037312040925688, 0.09312810253557502, 0.8450178531660164, 0.7708605980145241, 0.9174193222395155, 0.9360847915177604, 0.9728945553673997, 0.8924044813867645, 0.3344234547355259, 0.4074543686639025, 0.6045636063436775, 0.3023023162308862, 0.2705582083039584, 0.004467907427814777, 0.33949423973766546, 0.567792631377103, 0.6733832635263216, 0.9092464256125472, 0.6391266828808223, 0.8222586744278167, 0.4353056928231944, 0.32807687364058624, 0.5823867367269333, 0.33165248663801095, 0.1209076817785153, 0.4718271898635851, 0.893110465149153, 0.6080516527215041, 0.5660764749626511, 0.3685678574263902, 0.7873727430743175, 0.46823290788998784, 0.42839464994707843, 0.9660626394167192, 0.30941661645900853, 0.6404381982339297, 0.2183935736895367, 0.1603592624291228, 0.08837181419626938, 0.2487976345902635, 0.044053064332662095, 0.2000614354192075, 0.15526967953426485, 0.45147790697666257, 0.9135219784083205, 0.3658922819751893, 0.5625737358545079, 0.6129382891294904, 0.7814029253839128, 0.1811747142816137, 0.27719734589107536, 0.04602891321301583, 0.4605697464105697, 0.7730152408529295, 0.08186352836440758, 0.6322428524686667, 0.06485297798478373, 0.6926230904107078, 0.45055812534371764, 0.7662254525554929, 0.8251527384285235, 0.8898151405618799, 0.37877454773798536, 0.24223236197920128, 0.6419702738915475, 0.2750789126198281, 0.7403514258679206, 0.8780614794369656, 0.49214900498564296, 0.5950458670832343, 0.5867310699389676, 0.8823296597550807, 0.8978625724993561, 0.4332032074557779, 0.891874887248708, 0.480030368698966, 0.3715622808374748, 0.7065721309284982, 0.27500893915044844, 0.28270463524257583, 0.6156512506675406, 0.4502025800179047, 0.054333465681039916, 0.9782447054080851, 0.5771817946641751, 0.3963138567376957, 0.38487029723638966, 0.1398851090372134, 0.5733047420423418, 0.45409130678782594, 0.08537925258761558, 0.2866817743999971, 0.17695194230453248, 0.7758350154139825, 0.13449461365042592, 0.5635521968571511, 0.6356225196573024, 0.20817596288573248, 0.7814291451973491, 0.11631622007582398, 0.9490514551776089, 0.17648808415910144, 0.6701898754067173, 0.4778619954446879, 0.9991173437799168, 0.861418485132833, 0.3018726200374432, 0.6213130064464242, 0.47841172226175066, 0.6367541427527349, 0.8945935211723419, 0.5082039605474094, 0.527237071283252, 0.2961172828873183, 0.4943186040393148, 0.7320255584875159, 0.44199463201979183, 0.8806131464686789, 0.2074536717891614, 0.5131079510291038,0.4202067057337815, 0.9835350732001332, 0.8748409440560911, 0.4192125457417347, 0.27434141919278043, 0.45860298853846804, 0.4068242930747721, 0.3088887248044525, 0.2854244939898364, 0.08842639988534506, 0.8912040907950246, 0.18699479212639725, 0.33102860739553885, 0.15010863251822182, 0.24105270743775908, 0.15626715005277225, 0.1928580318522043, 0.7490219429882178, 0.11870277009233221, 0.5082883147898503, 0.79244618613363, 0.41182590890388315, 0.6558463912906267, 0.7216440573081493, 0.12185019361160532, 0.8985292657933432, 0.9512502953244601, 0.16671459069313777, 0.2037288357935172, 0.9851393062533914, 0.994706634360309, 0.1474552178342987, 0.45450117680905644, 0.712558673437983, 0.745510527857627, 0.6963828698206638, 0.10296513174669697, 0.2906265571381045, 0.06849249672015723, 0.9969524767543441, 0.4398336030464771, 0.37144576560104126, 0.13257202232053833, 0.8350197964151439, 0.018798594106191002, 0.13884493126050135, 0.242576830707247, 0.8617156016166727, 0.019122277877124505, 0.907500448069017,0.9315064109501823, 0.6440453617750412, 0.16427420194687037, 0.16035672991680194, 0.33331579338076167, 0.6081914605153201, 0.9183814537976289, 0.594429918861892, 0.372465126789847, 0.13382148257769322, 0.5042391321263469, 0.3257022033785367, 0.1194208133699397, 0.7569517803094309, 0.8349750444405716, 0.6655358639435216, 0.7358175315090151, 0.3400198462074022, 0.6117372685595409, 0.4219344477530955, 0.23494486508533008, 0.0884734751073244, 0.4487530664487078, 0.6207345956257121, 0.8592834981117227, 0.24090701589356844, 0.05045018697716219, 0.1591432456674574, 0.20707480759225705, 0.2339239298640915, 0.46579213736594427, 0.8661853742837542, 0.7182409282992872, 0.5600613370979511, 0.568589821474072, 0.21332530114088732, 0.5584279515071312, 0.993886752385122, 0.6839103490670199, 0.9801237902819316, 0.6117715483910731, 0.009451200438242435, 0.41219454491004337, 0.5384102730733391, 0.9586697119587726, 0.9813029704971501, 0.49851719435139674, 0.5143174230118772, 0.9088148207820242, 0.11758170138609203, 0.4467102586294809, 0.46457040002822325, 0.33550547582974055, 0.552133024272412, 0.10681807845184221, 0.34038537972454364, 0.7935212214143804, 0.13866789108962863, 0.9110150002755056, 0.11085883242254135, 0.07979371023446435, 0.49288306659684467, 0.9046938287207419, 0.4411166970439263, 0.2640793154759491, 0.07008939422924343, 0.7525357425485653, 0.13339173630981693, 0.5186009378858281, 0.7818384254458133, 0.05208379148147624, 0.19166488982907537, 0.27702409401216976, 0.8887633588570155, 0.03142593287519535, 0.00036326500247674254, 0.9111904150988731, 0.052227481595628955, 0.8290922770624574, 0.01501726124037639, 0.06230743956191753, 0.10069031706837972, 0.7168972583277471, 0.6943773502275037, 0.6900085244152514, 0.5572353532428568, 0.7762765123000817, 0.7424734623591849, 0.26672718355505565, 0.16386449487977162, 0.05563287602562195, 0.717964849462739, 0.6307039437866688, 0.3478764059541405, 0.04318437127441266, 0.4997692799379291, 0.3897782688789121, 0.455066694025977, 0.8785627582699149, 0.24060143812516288, 0.2455997105898613, 0.41607718861070964, 0.7326333083899966, 0.6244793819540186, 0.49510892633349635, 0.8995379324707836, 0.4760428921586858, 0.8248010399775206, 0.24600881926046936, 0.008055419736018532, 0.9449826140228736, 0.47936032012617347, 0.5546619179484152, 0.26950222675109825, 0.620749799497568, 0.4230730791354985, 0.6881989136391934, 0.9166308434732161, 0.0794650979820618, 0.3802930134400331, 0.5947351803524693, 0.6438345761756716, 0.5642759998690142, 0.44551622414266145, 0.10327410579487317, 0.9049392961003456, 0.3189783347300523, 0.801414798624298, 0.14194807998844483, 0.5777427734861819, 0.1615397089207139, 0.8263445593899176, 0.9507305427311054, 0.21926361032861386, 0.11628175884530423, 0.045429838853979665, 0.5085393203093713, 0.42844434600775594, 0.28672416458436867, 0.5512024369534446, 0.19239956930491253, 0.4489953148383722, 0.5851236049177956, 0.3168007981153299, 0.3595437270187517, 0.36060061358776885, 0.5874860450549293, 0.17338666251973267, 0.07553792250811064, 0.7328140005089139, 0.9481166006089077, 0.3687569063473404, 0.21071862232160732, 0.8547602843406643, 0.814390035254118, 0.4245522626185593, 0.9551270023706878, 0.8608218320873096, 0.00826300904579147, 0.14437257041619578, 0.6756170333775416, 0.43788201018822104, 0.1378866302722661, 0.9898683836178938, 0.20262275474257174, 0.2684059840542976, 0.9484542085376072, 0.5474603213776785, 0.5326543180834173, 0.2775426873369501, 0.7056951120354386, 0.7648946416536637, 0.030049764438252358, 0.6746576470496968, 0.47912823923919157, 0.5986326630968674, 0.9791711211851549, 0.028068557364871793, 0.7209147725526805, 0.9836222594817899, 0.8910995517037139, 0.7716898027974906, 0.05908309496575015, 0.5730437605128664, 0.7633102956310877, 0.9260379051134799, 0.7748684723600656, 0.07953520294514771, 0.8484544358250757, 0.3828280999774084, 0.796011351341852, 0.056069692503728175, 0.3924848058226239, 0.5524693937326164, 0.6701960326339607, 0.5802848733706302, 0.6347772582871473, 0.5564228390762354, 0.29193199937393166, 0.9295744354208131, 0.375730105833769, 0.4991874451034344, 0.12665715162275992, 0.16713165077209025, 0.3146694836292787, 0.07244528629620728, 0.190398752023612, 0.70173986983015, 0.9355455374737721, 0.2965187820144982, 0.6070702943730596, 0.5693441736509975, 0.8088630466482131, 0.05323164628296628, 0.9358199377644512, 0.4391194442250954, 0.5918369211964183, 0.30045330680427806, 0.5788151915291754, 0.7210272674916357, 0.6029467349339162, 0.4134672078181978, 0.636306052420017, 0.1030892697500948, 0.6060375528421117, 0.2625902016692129, 0.8811265527687406, 0.45693294649243055, 0.1591931535880713, 0.8675341783648784, 0.025209927111295105, 0.030317282440130766, 0.26312332022858764, 0.4797977926790149, 0.7942814570782198, 0.37440942347066497, 0.4210362835773428, 0.9664352038531806, 0.31363248161543067, 0.369813645046844, 0.8276125141892737, 0.10847508362607516, 0.8562120708252523, 0.5727732862157318, 0.9692447027904683, 0.3388342164004121, 0.42027365315531273, 0.29116949077776877, 0.8852124565647926, 0.4177824079479059, 0.5324402729795684, 0.8214020222177698, 0.1911369612925078, 0.13547300651522398, 0.8069274852998902, 0.17712099173942275, 0.08722173372769904,0.3282125331097573, 0.6843382234770045, 0.11154732132270284, 0.20242736528671834, 0.6378823844058699, 0.9910951605902675, 0.948058533885942, 0.5919650387013952, 0.04428975505595745, 0.07650746564163013, 0.9121925331230927, 0.9240705016340519, 0.22770780269320867, 0.41854014604496714, 0.48578523445773414, 0.33193710282654987, 0.7903586907094168, 0.6697173836526329, 0.2049034334535912, 0.18730709240595478, 0.5855314257458567, 0.471242961037206, 0.47512712166713533, 0.20559212955401152, 0.5481149604629276, 0.12414994499627974, 0.4514219623716911, 0.5853260800545231, 0.7882466936254782, 0.584870948802918, 0.31124560962678993, 0.752293606314185, 0.37027423326509157, 0.9207443075867916, 0.2703416570074161, 0.02022468333316818, 0.24370470394845456, 0.2483926947208167, 0.3312513290532948, 0.3837575747922185, 0.5296926770262497, 0.1787463117675211, 0.8227917450688984, 0.9285466341876488, 0.4729096735766888, 0.40397489487972094, 0.0399611938171377, 0.35044459866260536, 0.8621707088368072, 0.9482927209475428, 0.7203070352237586, 0.019182101473728674, 0.6087150737673583, 0.2532175080672351, 0.6036682357781489, 0.9835989358870332, 0.26905784034368874, 0.21025111361352888, 0.8143646504625376, 0.21808517821324946, 0.9541682652747587, 0.9236308780358258, 0.6020039755429897, 0.39392517602602817, 0.9090720646034769, 0.2978578966056161, 0.8986613874703445, 0.41966725943188987, 0.39277702473596265, 0.8188637572712868, 0.5890165026760947, 0.6700967630811683, 0.9415857537234482, 0.6140634745899282, 0.1737620004576882, 0.7023131876631714, 0.4620606016689779, 0.38919497386851387, 0.2938722123597234, 0.48851830481452907, 0.08996794602391378, 0.26569256555741816, 0.5609906920446511, 0.3914555455842894, 0.05755916579474263, 0.6695550877800012, 0.43075237561635227, 0.1709184912501045, 0.5199468192013912, 0.5158975022582966, 0.2674943383847431, 0.01731042345424505, 0.7767763775155538, 0.7886262632763167, 0.7101794957875919, 0.2770103525138511, 0.10227548545030452, 0.3221443721384951, 0.5927253620962482, 0.96025734535449, 0.015738952957527808, 0.6170590400080567, 0.6453063003494637, 0.3768251905126073, 0.13256465510968873, 0.554295348419922, 0.456983267037937, 0.7145284076998268, 0.036095199542039325, 0.6794676876331511, 0.5553780174601407, 0.7966783558795584, 0.7203737132894587, 0.9241424429298525, 0.8739856281118867, 0.39622462272775727, 0.9680240492976241, 0.03188796193636101, 0.2998947762934596, 0.2255741384480806, 0.060496394424358724, 0.9827847113457324, 0.18794075526996634, 0.3727776074214496, 0.29587234484578784, 0.4494477182644486, 0.2766300761127224, 0.6096410201973025, 0.19166002142305838, 0.02188271315708512, 0.16223709781884554, 0.1316623682322291, 0.8775392625238125, 0.7591936415790552, 0.4998710983327401, 0.9931197381556702, 0.9774145566025858, 0.7360905508242229, 0.5287899498457144, 0.9037752698327245, 0.42730071856403495, 0.9154225053552859, 0.8879914301784473, 0.518542788940781, 0.655298421463517, 0.7401702345726874, 0.2032246983434064, 0.12158779942043785, 0.5101955951147246, 0.6889031280605671, 0.24705901949186215, 0.23871707505713502, 0.402591923704543, 0.1554961557078175, 0.05260643899879802, 0.03686192750004913, 0.20496026873019735, 0.387359465587264, 0.09366273035474049, 0.7100310003712904, 0.6655798114054116, 0.8144782635088726, 0.823591485281024, 0.5767851510185626, 0.2340050421633426, 0.8988086669585299, 0.939237537008749, 0.01035567055634845, 0.7346257971905314, 0.783091325731317, 0.3070598072766234, 0.03205415151464064, 0.4789039790155796, 0.7788707409448673, 0.19748135406672118, 0.44433639851344564, 0.711851685086178, 0.5195821007775707, 0.047307281201704465, 0.5632959762645935, 0.23685100697435357, 0.44798156818259927, 0.8435413237546837, 0.3581260615451425, 0.026374492945924977, 0.4343044291962541,0.4700531636921641, 0.4534084193118523, 0.7168286216581272, 0.4240154719219562, 0.6969257514416167, 0.47113300660196644, 0.3481039755384545, 0.033960329389228994, 0.7928333180808108, 0.7395316936685102, 0.24533262064324846, 0.16038532880535628, 0.26948042384494386, 0.7362776254570321, 0.3952497718429058, 0.9008515889479082, 0.047686879409277294, 0.48065389615063914, 0.7574908273230464, 0.6418620391694914, 0.5761070764519093, 0.14035180908318678, 0.4808296966296469, 0.3792333711748187, 0.14004971978011238, 0.4051352435012089, 0.7756857922606376, 0.21723905665983545, 0.8748375058259324, 0.8584416414217928, 0.646810743939829, 0.9352241355721943, 0.8163913288454224, 0.09455563866471572, 0.5997571107964288, 0.20163185855675747, 0.04949906199789833, 0.3769771242067659, 0.8627441827254738, 0.7474402935438827, 0.3717382152735983, 0.5139470709752325, 0.9073461050325751, 0.5294342510730735, 0.14482034661899423, 0.5697926130559755, 0.8923839152964801, 0.879909112192456, 0.9890350760685254, 0.30336402899800197,0.28766556192496406, 0.4119083798949308, 0.16142922424118555, 0.34791615035186296, 0.15430949963985174, 0.2130466012716068, 0.5676552275701336, 0.18820802118347435, 0.23124679682152394, 0.7279197374548184, 0.7666090005183201, 0.6344422702670374, 0.013749400673376155, 0.8481905464184771, 0.5059256200463839, 0.4899793684857848, 0.6601821811630284, 0.7983245565976883, 0.8277325644156374, 0.9180577456220427, 0.29791090413933785, 0.4959855916437983, 0.0148612979619005, 0.20915169806156753, 0.37771254023849377, 0.33049208247210027, 0.22564418843888645, 0.1480994870346759, 0.16455393863460455, 0.007931291935408202, 0.7302059454745164, 0.42903556673714305, 0.35725097698434993, 0.8177321633681842, 0.9736383696028198, 0.11426692623997492, 0.7121312390228457, 0.8319351893271613, 0.909472149464639, 0.6737049485292633, 0.19633438758360178, 0.15576909234596192, 0.5034233813592887, 0.46176079363015865, 0.6627310851135377, 0.11179455475870503, 0.42186688393924, 0.14242954109132477, 0.6903486592821282, 0.09753207457231683, 0.8831260225865442, 0.7424496573093403, 0.7403525943803368, 0.23676769340878923, 0.5143149024510068, 0.22146524679567692, 0.6181278145353054, 0.7188409682293436, 0.18993756357242453, 0.02003695946911388, 0.9206097737209655, 0.2443119876162081, 0.8281603600779406, 0.9185374030994518, 0.551335281592804, 0.3432733032351125, 0.963091636181615, 0.5005947583190296, 0.8212769091605224, 0.15792832545713764, 0.01368447324799904, 0.9434544936441975, 0.276382329981842, 0.18760171633195566, 0.3799637849490649, 0.3215466817687276, 0.21155822423251147, 0.15041699872526693, 0.08566221291100218, 0.9201501001531511, 0.7579485695872292, 0.6034908044408858, 0.4600157872408064, 0.006077874608385603, 0.8431610233136543, 0.10619599413230607, 0.7192644073291087, 0.26750427012770195, 0.9274032367845411, 0.46133276684470825, 0.4586866762274089, 0.3496644153077756, 0.560869005105939, 0.8937703291834528, 0.1983172944850785, 0.30391037897794715, 0.2745317954773452, 0.4129595179595751, 0.14524292162420827, 0.9533407298375065, 0.7900187457440871, 0.09745980161430168, 0.017365203435852905, 0.04571841695407952, 0.583540086382026, 0.22375471310663186, 0.36282970160004935, 0.7204793531099422, 0.5495993470541658, 0.42573017158991755, 0.9298099377316281, 0.44272273721098343, 0.5952657360263481, 0.25454682537166307, 0.03896925493151471, 0.9819754085195709, 0.9014094643174677, 0.08624448497266246, 0.5316241661088743, 0.10724232856162452, 0.5845765064240795, 0.012588724417248787, 0.13725776432257342, 0.34694525642415286, 0.13028011830581643, 0.2551384822150755, 0.8832931173345753, 0.3858970900379247, 0.7583801075261248, 0.17895952398768167, 0.04441328327047256, 0.45139868312300746, 0.25336449945198025, 0.11456095610724049, 0.018827050341823637, 0.8513432960865948, 0.6788217046438484, 0.5370881408769471, 0.30183131237723004, 0.7864449756148978, 0.9307865359607256, 0.2905135067460368, 0.6554208983954675, 0.1147747389854995, 0.5940296874937254, 0.874534608441179, 0.08252168796021231, 0.46725059959269544, 0.9291790458341708, 0.025560882260210804, 0.9707568686232968, 0.41427370090372695, 0.9623408387010625, 0.18027502071653567, 0.8941602656131795, 0.2323728300736323, 0.17048112221155476, 0.4590035221409158, 0.1555266300231516, 0.5105857586935169, 0.3683336794085179, 0.8130350895001692, 0.029171698950285596, 0.6993722788554764, 0.16292881798574976, 0.39940867681601055, 0.7942489923890919, 0.41822876397944353, 0.6355622016874548, 0.16409745069350123, 0.004182693523884451, 0.8054035016466043, 0.25533473973629695, 0.13228439304536665, 0.29822877021828753, 0.6458964732076087, 0.8470192070603739, 0.3908559731947733, 0.07145843633188009, 0.9866599044520856, 0.8853914776388428, 0.6996800036158664, 0.7144469404048177, 0.23039143101127935, 0.6910049779268469, 0.0753585699142083, 0.1801388706367042, 0.9173900734163123, 0.22926423214634262, 0.7006607158058921, 0.11990126943039303, 0.29112622315169245, 0.6775690268473594, 0.9638190558197468, 0.02809676151606122, 0.4105389550940328, 0.7971128497820327, 0.12038844831443463, 0.07430695018135469, 0.3597796554047715, 0.8406274614039912, 0.19981211386920872, 0.986908315414069, 0.4833352611747592, 0.4712917680405636, 0.39747649324046763,0.11459748239801504, 0.05972190823846624, 0.048060180055336854, 0.6227511502552616, 0.9649336178857377, 0.22335424046310903, 0.11174522671237053, 0.0176103664811339, 0.47173117837261525, 0.2539759725452223, 0.6574738296108226, 0.09941280424839727, 0.9249770647878726, 0.9016075396289763, 0.26125458071456575, 0.8252892801163306, 0.0989530747021935, 0.0397220306206818, 0.9629659540847427, 0.3561324813915072, 0.560897862084912, 0.29421988042984126, 0.5943930851565672, 0.35746236391386954, 0.04493609978704716, 0.906977095512643, 0.255727436175042, 0.7930780770572305, 0.4540280216224911, 0.46275705527377964, 0.6812451709581524, 0.3395087299860927, 0.02179971487709731, 0.4033770205269366, 0.21730790948401302, 0.13810938850616916, 0.8832816825111316, 0.7227398478829137, 0.2569596788559395, 0.6221568427445674, 0.8064002915453888, 0.2830079169282541, 0.48169379719502015, 0.6066893842138618, 0.48955714057855326, 0.7498238839193883, 0.2673564918376511, 0.9536767447013428, 0.633403549381697, 0.35719498512331205,0.5747537418875732, 0.27874383061600516, 0.30136775473025224, 0.792347343720588, 0.34054762236588965, 0.12360629767220377, 0.6548070946984367, 0.35694880032884424, 0.3048649285716204, 0.5719451855218745, 0.055960669128767404, 0.07695403586917882, 0.7259164361310297, 0.4827237055463448, 0.7420094188768105, 0.17475894974053097, 0.1930384638365622, 0.8489980909394655, 0.3154038909740474, 0.5116712430269631, 0.20188268837434742, 0.3757309924374299, 0.01183000057092809, 0.6824580503925004, 0.520447310073781, 0.3185635161466007, 0.5717416246082737, 0.32425728169396184, 0.5455566798452368, 0.9973463302557554, 0.3600831992603104, 0.5642265714916832, 0.14547917954425071, 0.24924939520680167, 0.8499084991071096, 0.11305579311423919, 0.21660446318607085, 0.9292981858710375, 0.02713801080691336, 0.16958736369849992, 0.6294586347224489, 0.20766842941829222]

# or

# import random

# input_embedding = [random.random() for _ in range(1536)]

# # print(vector)
# print("Length:", len(input_embedding))


In [ ]:
results = await embeddings_client.search_in_collection(
    collection_name="test_hist",
    input=input_embedding,
    top_k=5,
    distance_metric="cosine_similarity",
)

df = pd.json_normalize(results["embeddings"])
df

In [ ]:
results = await embeddings_client.search_in_collection(
    collection_name="test_hist",
    input=input_embedding,
    top_k=5,
    distance_metric="cosine_similarity",
    filters={"model": "hist"},
    max_value=0.5,
    min_value=0.01,
    exclude_info = True,
)

df = pd.json_normalize(results["embeddings"])
df

In [ ]:
results = await embeddings_client.search_in_collection(
    collection_name="test_hist",
    input=input_embedding,
    top_k=5,
    distance_metric="l1_distance",
)

df = pd.json_normalize(results["embeddings"])
df

In [ ]:
results = await embeddings_client.search_across_collections(
    input=input_embedding,
    top_k=10,
    distance_metric="inner_product",
)

df = pd.json_normalize(results["embeddings"])
df